In [ ]:
import os
from dotenv import load_dotenv
#
load_dotenv()

#
# 1. Configure authentication credentials
# If running inside a Databricks Notebook, the host and token are automatically discovered.
DATABRICKS_HOST = os.environ.get("DATABRICKS_HOST")
DATABRICKS_TOKEN = os.environ.get("DATABRICKS_PERSONAL_ACCESS_TOKEN")
WAREHOUSE_ID = os.environ.get("DATABRICKS_WAREHOUSE_ID")  # Found under SQL Warehouses -> Connection details


In [ ]:
from databricks import sql
import os

#
connection = sql.connect(
                server_hostname = DATABRICKS_HOST,
                http_path = f"/sql/1.0/warehouses/{WAREHOUSE_ID}",
                access_token = DATABRICKS_TOKEN
            )

cursor = connection.cursor()

cursor.execute("SELECT * from range(10)")
print(cursor.fetchall())

cursor.close()
connection.close()

[Row(id=0), Row(id=1), Row(id=2), Row(id=3), Row(id=4), Row(id=5), Row(id=6), Row(id=7), Row(id=8), Row(id=9)]


Set User-Agent

In [ ]:
from databricks import sql
import os

with sql.connect(server_hostname   = os.getenv("DATABRICKS_SERVER_HOSTNAME"),
                 http_path         = os.getenv("DATABRICKS_HTTP_PATH"),
                 access_token      = os.getenv("DATABRICKS_TOKEN"),
                 user_agent_entry = "product_name") as connection:
  with connection.cursor() as cursor:
    cursor.execute("SELECT 1 + 1")
    result = cursor.fetchall()

    for row in result:
      print(row)



Query data

In [ ]:
from databricks import sql
import os

with sql.connect(server_hostname = os.getenv("DATABRICKS_SERVER_HOSTNAME"),
                 http_path       = os.getenv("DATABRICKS_HTTP_PATH"),
                 access_token    = os.getenv("DATABRICKS_TOKEN")) as connection:

  with connection.cursor() as cursor:
    cursor.execute("SELECT * FROM samples.nyctaxi.trips LIMIT ?", [2])
    result = cursor.fetchall()

    for row in result:
      print(row)

Query tags

In [ ]:
# Session-level tags:

from databricks import sql
import os

with sql.connect(
    server_hostname = os.getenv("DATABRICKS_SERVER_HOSTNAME"),
    http_path       = os.getenv("DATABRICKS_HTTP_PATH"),
    access_token    = os.getenv("DATABRICKS_TOKEN"),
    query_tags = {"team": "engineering", "dashboard": "abc123", "env": "prod"}
) as connection:
    with connection.cursor() as cursor:
        cursor.execute("SELECT * FROM samples.nyctaxi.trips LIMIT ?", [2])
        result = cursor.fetchall()
        for row in result:
            print(row)

In [ ]:
# Statement-level tags:
from databricks import sql
import os

with sql.connect(
    server_hostname = os.getenv("DATABRICKS_SERVER_HOSTNAME"),
    http_path       = os.getenv("DATABRICKS_HTTP_PATH"),
    access_token    = os.getenv("DATABRICKS_TOKEN"),
) as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            "SELECT * FROM samples.nyctaxi.trips LIMIT ?",
            parameters=[2],
            query_tags={"team": "engineering", "dashboard": "abc123", "env": "prod"}
        )
        result = cursor.fetchall()
        for row in result:
            print(row)

Insert data

In [ ]:
from databricks import sql
import os

with sql.connect(server_hostname = os.getenv("DATABRICKS_SERVER_HOSTNAME"),
                 http_path       = os.getenv("DATABRICKS_HTTP_PATH"),
                 access_token    = os.getenv("DATABRICKS_TOKEN")) as connection:

  with connection.cursor() as cursor:
    cursor.execute("CREATE TABLE IF NOT EXISTS squares (x int, x_squared int)")

    squares = [(i, i * i) for i in range(100)]

    cursor.executemany("INSERT INTO squares VALUES (?, ?)", squares)

    cursor.execute("SELECT * FROM squares LIMIT ?", [10])

    result = cursor.fetchall()

    for row in result:
      print(row)

Query metadata

In [ ]:
from databricks import sql
import os

with sql.connect(server_hostname = os.getenv("DATABRICKS_SERVER_HOSTNAME"),
                 http_path       = os.getenv("DATABRICKS_HTTP_PATH"),
                 access_token    = os.getenv("DATABRICKS_TOKEN")) as connection:

  with connection.cursor() as cursor:
    cursor.columns(schema_name="default", table_name="squares")
    print(cursor.fetchall())

#### Manage cursors and connections

It is a best practice to close any connections and cursors that are no longer in use. This frees resources on Databricks all-purpose compute and Databricks SQL warehouses.

You can use a context manager (the with syntax used in previous examples) to manage the resources, or explicitly call close:

In [ ]:
from databricks import sql
import os

connection = sql.connect(server_hostname = os.getenv("DATABRICKS_SERVER_HOSTNAME"),
                         http_path       = os.getenv("DATABRICKS_HTTP_PATH"),
                         access_token    = os.getenv("DATABRICKS_TOKEN"))

cursor = connection.cursor()

cursor.execute("SELECT * from range(10)")
print(cursor.fetchall())

cursor.close()
connection.close()

##### Manage files in Unity Catalog volumes

The Databricks SQL Connector enables you to write local files to Unity Catalog volumes, download files from volumes, and delete files from volumes, as shown in the following example:

In [ ]:
from databricks import sql
import os

# For writing local files to volumes and downloading files from volumes,
# you must set the staging_allowed_local_path argument to the path to the
# local folder that contains the files to be written or downloaded.
# For deleting files in volumes, you must also specify the
# staging_allowed_local_path argument, but its value is ignored,
# so in that case its value can be set for example to an empty string.
with sql.connect(server_hostname            = os.getenv("DATABRICKS_SERVER_HOSTNAME"),
                 http_path                  = os.getenv("DATABRICKS_HTTP_PATH"),
                 access_token               = os.getenv("DATABRICKS_TOKEN"),
                 staging_allowed_local_path = "/tmp/") as connection:

  with connection.cursor() as cursor:

    # Write a local file to the specified path in a volume.
    # Specify OVERWRITE to overwrite any existing file in that path.
    cursor.execute(
      "PUT '/tmp/my-data.csv' INTO '/Volumes/main/default/my-volume/my-data.csv' OVERWRITE"
    )

    # Download a file from the specified path in a volume.
    cursor.execute(
      "GET '/Volumes/main/default/my-volume/my-data.csv' TO '/tmp/my-downloaded-data.csv'"
    )

    # Delete a file from the specified path in a volume.
    cursor.execute(
      "REMOVE '/Volumes/main/default/my-volume/my-data.csv'"
    )

#### Configure logging
The Databricks SQL Connector uses Python's standard logging module. The following example configures the logging level and generates a debug log:

In [ ]:
from databricks import sql
import os, logging

logging.getLogger("databricks.sql").setLevel(logging.DEBUG)
logging.basicConfig(filename = "results.log",
                    level    = logging.DEBUG)

connection = sql.connect(server_hostname = os.getenv("DATABRICKS_SERVER_HOSTNAME"),
                         http_path       = os.getenv("DATABRICKS_HTTP_PATH"),
                         access_token    = os.getenv("DATABRICKS_TOKEN"))

cursor = connection.cursor()

cursor.execute("SELECT * from range(10)")

result = cursor.fetchall()

for row in result:
   logging.debug(row)

cursor.close()
connection.close()

In [ ]:

from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import create_sql_agent
from langchain_openai import ChatOpenAI
from langchain_community.agent_toolkits import SQLDatabaseToolkit

# 2. Initialize the Databricks SQL database connection wrapper
db = SQLDatabase.from_databricks(
    catalog="samples",
    schema="nyctaxi",
    host=DATABRICKS_HOST,
    api_token=DATABRICKS_TOKEN,
    warehouse_id=WAREHOUSE_ID,
    #include_tables=[]
)

# 3. Instantiate your LLM of choice
llm = ChatOpenAI(model="gemma4:latest", temperature=0)

# 4. Construct the toolkit and the agent executor
toolkit = SQLDatabaseToolkit(db=db, llm=llm)
agent_executor = create_sql_agent(
    llm=llm,
    toolkit=toolkit,
    verbose=True,
    agent_type="openai-tools"
)

# 5. Run a natural language query
response = agent_executor.invoke({"input": "What was the average fare amount for trips in 2016 ?"})
print(response["output"])


[WARN] Parameter '_user_agent_entry' is deprecated; use 'user_agent_entry' instead. This parameter will be removed in the upcoming releases.




> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{'tool_input': ''}`


trips
Invoking: `sql_db_schema` with `{'table_names': 'trips'}`



CREATE TABLE trips (
	tpep_pickup_datetime TIMESTAMP, 
	tpep_dropoff_datetime TIMESTAMP, 
	trip_distance DOUBLE, 
	fare_amount DOUBLE, 
	pickup_zip INT, 
	dropoff_zip INT
) USING DELTA
TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'enabled')

/*
3 rows from trips table:
tpep_pickup_datetime	tpep_dropoff_datetime	trip_distance	fare_amount	pickup_zip	dropoff_zip
2016-02-13 21:47:53+00:00	2016-02-13 21:57:15+00:00	1.4	8.0	10103	10110
2016-02-13 18:29:09+00:00	2016-02-13 18:37:23+00:00	1.31	7.5	10023	10023
2016-02-06 19:40:58+00:00	2016-02-06 19:52:32+00:00	1.8	9.5	10001	10018
*/
Invoking: `sql_db_query_checker` with `{'query': "SELECT AVG(fare_amount) FROM trips WHERE STRFTIME('%Y', tpep_pickup_datetime) = '2016'"}`


SELECT AVG(fare_amount) FROM trips WHERE YEAR(tpep_pickup_datetime) = 2016
Invoking: `sql_db_quer

In [ ]:
from databricks import sql
import os

with sql.connect(server_hostname = os.getenv("DATABRICKS_SERVER_HOSTNAME"),
                 http_path       = os.getenv("DATABRICKS_HTTP_PATH"),
                 access_token    = os.getenv("DATABRICKS_TOKEN")) as connection:

  with connection.cursor() as cursor:
    cursor.execute("CREATE TABLE IF NOT EXISTS squares (x int, x_squared int)")

    squares = [(i, i * i) for i in range(100)]

    cursor.executemany("INSERT INTO squares VALUES (?, ?)", squares)

    cursor.execute("SELECT * FROM squares LIMIT ?", [10])

    result = cursor.fetchall()

    for row in result:
      print(row)

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks_langchain import ChatDatabricks, DatabricksMCPServer, DatabricksMultiServerMCPClient
from langchain.agents import create_agent
from com.example.ai.LLMManager import LLMManager
 
workspace_client = WorkspaceClient()
host = workspace_client.config.host

mcp_client = DatabricksMultiServerMCPClient([
    DatabricksMCPServer(
        name="uc-functions",
        url=f"{host}/api/2.0/mcp/functions/workspace/default",
        workspace_client=workspace_client,
    ),
])

async with mcp_client:
    tools = await mcp_client.get_tools()
    agent = create_agent(
        ChatDatabricks(endpoint="databricks-claude-sonnet-4-5"),
        tools=tools,
    )
    result = await agent.ainvoke(
        {"messages": [{"role": "user", "content": "Look up customer info for Acme Corp"}]}
    )
    print(result["messages"][-1].content)



NotImplementedError: As of langchain-mcp-adapters 0.1.0, MultiServerMCPClient cannot be used as a context manager (e.g., async with MultiServerMCPClient(...)). Instead, you can do one of the following:
1. client = MultiServerMCPClient(...)
   tools = await client.get_tools()
2. client = MultiServerMCPClient(...)
   async with client.session(server_name) as session:
       tools = await load_mcp_tools(session)

In [ ]:
import asyncio
from databricks.sdk import WorkspaceClient
from databricks_mcp import DatabricksMCPClient
#from langchain_mcp_adapters.tools import convert_mcp_tool # Or pass via MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI

async def main():
    # 1. Initialize the official Databricks Workspace SDK
    workspace_client = WorkspaceClient()
    host = workspace_client.config.host

    # 2. Point to your specific Databricks MCP Server URL
    # (Example shows a Managed Unity Catalog function endpoint)
    server_url = f"{host}/api/2.0/mcp/functions/workspace/default"
    
    mcp_client = DatabricksMCPClient(
        server_url=server_url,
        workspace_client=workspace_client,
    )

    # 3. Pull the live schema tools from Databricks
    # Databricks native client returns tools which can be mapped directly
    mcp_tools = mcp_client.list_tools()
    
    # 4. Convert the fetched MCP tools to LangChain/LangGraph style
    # Alternatively, you can feed the server params into MultiServerMCPClient 
    from langchain_mcp_adapters.client import MultiServerMCPClient
    
    # Using LangChain's MultiServer client via an SSE/HTTP link configuration:
    async with MultiServerMCPClient(
        {
            "databricks_tools": {
                "url": f"{server_url}/sse", # Databricks applications expose tools over SSE
                "transport": "sse",
                "headers": {"Authorization": f"Bearer {workspace_client.config.token}"}
            }
        }
    ) as langchain_mcp_client:
        
        # 5. Extract the tools
        langchain_tools = langchain_mcp_client.get_tools()

        # 6. Bind them to your LangChain Agent workflow
        model = ChatOpenAI(model="gpt-4o")
        agent = create_react_agent(model, langchain_tools)

        # 7. Query your agent 
        response = await agent.ainvoke(
            {"messages": [("user", "Run the calculation using the Unity Catalog function.")]}
        )
        print(response["messages"][-1].content)

if __name__ == "__main__":
    asyncio.run(main())


In [ ]:
from databricks_langchain import UCFunctionToolkit
#from langgraph.prebuilt import create_react_agent
from langchain.agents import create_agent
from com.example.ai.LLMManager import LLMManager

# Pass the absolute path of your function directly into LangChain
toolkit = UCFunctionToolkit(function_names=["workspace.default.add_numbers"])
langchain_tools = toolkit.tools

model = LLMManager.get_model()

# Deploy instantly inside your agent structure
agent = create_agent(model, langchain_tools)
